In [18]:
import numpy as np
import pandas as pd
import os
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader


In [19]:
transform=transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [20]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['id_code'] + '.png')
        image = Image.open(img_path).convert('RGB')
        label = row['diagnosis']
        if self.transform:
            image = self.transform(image)
        return image, label


In [21]:
train_data = BlindnessDataset(
    csv_file='data/raw/aptos2019-blindness-detection/train.csv',
    img_dir='data/raw/aptos2019-blindness-detection/train_images',
    transform=transform
)

test_data = BlindnessDataset(
    csv_file='data/raw/aptos2019-blindness-detection/test.csv',
    img_dir='data/raw/aptos2019-blindness-detection/test_images',
    transform=transform
)

train_loader = torch.utils.data.DataLoader(train_data, batch_size = 32, shuffle=True, num_workers=0, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size = 32, shuffle=True, num_workers=0, pin_memory=True)

In [22]:
image, label = train_data[0]
image.size()


torch.Size([3, 224, 224])

In [23]:
class_names = ['0 - No DR', '1 - Mild', '2 - Moderate', '3 - Severe', '4 - Proliferative DR']

In [24]:
class NeuralNet(nn.Module):

    def __init__(self):
        super().__init__()

        #Convolutional layer 1
        self.conv1 = nn.Conv2d(3, 16, 5) #1st: # of input channels, 2nd: # of feature maps filters, 3rd: filter size (5 by 5) (224 - 5 / 1) + 1 = 220, so (16, 220, 220)
        
        #Pooling
        self.pool = nn.AvgPool2d(2,2) #Picked Avg

        #Convolutional layer 2
        self.conv2 = nn.Conv2d(16, 32, 5) # (32, 106, 106) => 110 - 5 + 1 = 106
        
        #Flatten
        self.fc1 = nn.Linear(32 * 26 * 26, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 5)

    def forward(self, x):
        #F.relu is just Max(0, x), never negative
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(x) 
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


        

In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)  # should print "cuda"

net = NeuralNet().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

print(device)
print(next(net.parameters()).device)

cuda
cuda
cuda:0


In [28]:
import time

In [ ]:

for epoch in range(5):
    print(f'Training epoch/round {epoch}...')
    start = time.time()
    running_loss = 0.0

    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        if i % 10 == 0:
            print(f'Epoch {epoch} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}')

    print(f'Epoch {epoch} done | Avg Loss: {running_loss/len(train_loader):.4f} | Time: {time.time()-start:.1f}s')

Training epoch/round 0...
Epoch 0 | Batch 0/115 | Loss: 1.0715
Epoch 0 | Batch 10/115 | Loss: 1.0960
Epoch 0 | Batch 20/115 | Loss: 1.0726
Epoch 0 | Batch 30/115 | Loss: 1.1717
Epoch 0 | Batch 40/115 | Loss: 1.3315
